In [2]:
from transformers import BertTokenizer, BartForConditionalGeneration, Text2TextGenerationPipeline

tokenizer = BertTokenizer.from_pretrained("fnlp/bart-base-chinese")
model = BartForConditionalGeneration.from_pretrained("fnlp/bart-base-chinese")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/479 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/259k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/561M [00:00<?, ?B/s]

In [3]:
import pandas as pd
from transformers import BertTokenizer
from datasets import Dataset

train_df = pd.read_csv("train.csv")
train_df

,ques_content,ques_knowledges
0,题目内容相同的资讯对每个人的价值是相等的,信息的概念及其基本特征
1,题目内容请判断以下说法是否正确TF1人类历史上的第一次信息技术革命是通过文字的使用标志的,信息技术的发展
2,以下哪一组颜色属于中性色可以与任意颜色搭配因此也被称为通用色选项A黑白灰B红蓝绿C紫蓝青D赤橙黄,图像处理
3,从第三张幻灯片起每张幻灯片都需要加入一个自定义动作按钮该按钮需设置超链接至第二张幻灯片在幻灯...,演示文稿制作
4,假设有三个U盘分别为XYZ其容量为3GB3072KB15GB存储容量最大的U盘是盘,信息与信息技术基础
...,...,...
1187,题目内容请判断以下陈述的正误1输入和输出设备用于保存程序和数据,计算机系统
1188,题目内容请判断以下说法的对错TF1Excel2003是一个专门用来制作报表的工具而Word2...,Word表格处理
1189,题目内容朱红需要通过电子邮件将一篇文章发送给王明请选择正确的邮箱地址选项Azhuhong04...,电子邮件
1190,题目内容请写出以下计算机相关术语的英文缩写1本地网络2处理器3大范围网络4系统软件5智能系统,计算机基础


In [4]:
import pandas as pd
from transformers import BertTokenizer
from datasets import Dataset

test_df = pd.read_csv("test.csv")

# 加载分词器
tokenizer = BertTokenizer.from_pretrained("fnlp/bart-base-chinese")

# 数据预处理函数
def preprocess_function(examples):
    inputs = examples["ques_content"]
    targets = examples["ques_knowledges"]

    # 对输入和目标进行编码
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=128, truncation=True, padding="max_length")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# 将数据转换为 Hugging Face 数据集
train_dataset = Dataset.from_pandas(train_df).map(preprocess_function, batched=True)
test_dataset = Dataset.from_pandas(test_df).map(preprocess_function, batched=True)

Map:   0%|          | 0/1192 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:3953: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/299 [00:00<?, ? examples/s]

In [5]:
from transformers import BartForConditionalGeneration, Trainer, TrainingArguments

# 加载模型
model = BartForConditionalGeneration.from_pretrained("fnlp/bart-base-chinese")

# 训练参数
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

# 开始训练
trainer.train()

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:

 ··········


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


Epoch,Training Loss,Validation Loss
1,0.020400,0.010212
2,0.009300,0.008021
3,0.008300,0.007139


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:2817: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_eos_token_id': 102}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=447, training_loss=0.2580013050962348, metrics={'train_runtime': 372.9797, 'train_samples_per_second': 9.588, 'train_steps_per_second': 1.198, 'total_flos': 1090208787333120.0, 'train_loss': 0.2580013050962348, 'epoch': 3.0})

In [9]:
import os
from transformers import BartForConditionalGeneration

# 获取最新的 checkpoint 目录
results_dir = "./results"
latest_checkpoint = max(
    [os.path.join(results_dir, d) for d in os.listdir(results_dir) if d.startswith("checkpoint")],
    key=os.path.getmtime
)

# 加载最新的 checkpoint
model = BartForConditionalGeneration.from_pretrained(latest_checkpoint)
print(f"成功加载模型：{latest_checkpoint}")

成功加载模型：./results/checkpoint-447


In [10]:
from transformers import BertTokenizer, BartForConditionalGeneration

# 加载分词器
tokenizer = BertTokenizer.from_pretrained("fnlp/bart-base-chinese")

In [11]:
model.eval()

BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(51271, 768, padding_idx=0)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(51271, 768, padding_idx=0)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 768)
      (layers): ModuleList(
        (0-5): 6 x BartEncoderLayer(
          (self_attn): BartSdpaAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (final_lay

In [14]:
def predict(text):
    # 对输入进行编码（移除 token_type_ids）
    inputs = tokenizer(text, return_tensors="pt", max_length=512, truncation=True)

    # 只保留 model 需要的参数
    inputs = {k: v for k, v in inputs.items() if k in ["input_ids", "attention_mask"]}

    # 生成预测
    output_ids = model.generate(**inputs, max_length=128, num_beams=5)

    # 解码预测结果
    predicted_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return predicted_text

In [15]:
test_text = "以下哪一组颜色属于中性色可以与任意颜色搭配因此也被称为通用色选项A黑白灰B红蓝绿C紫蓝青D赤橙黄"
prediction = predict(test_text)
print("预测的知识点类型编号:", prediction)

预测的知识点类型编号: 图 像 处 理
